# NB-3 — Blocks B and C sanity check

Validate the association-risk and regime-mining blocks in isolation before they are
combined into the main results. Attach `halo-stage2`.

**Publish output as `halo-stage3`.**


In [ ]:
# --- HALO bootstrap -------------------------------------------------------------
# Attach these datasets to this notebook before running (right panel -> Add Data):
#   1. Competition: "ieee-fraud-detection"      (accept the rules first)
#   2. Your source dataset: "halo-src"           (the halo/ package, see RUN_GUIDE.md)
#   3. For stages after NB-1: the previous stage's output dataset
import os, shutil, sys, subprocess, time

SRC = "/kaggle/input/halo-src"
if os.path.exists(SRC):
    if os.path.exists("/kaggle/working/halo"):
        shutil.rmtree("/kaggle/working/halo")
    shutil.copytree(os.path.join(SRC, "halo"), "/kaggle/working/halo")
sys.path.insert(0, "/kaggle/working")

# Carry forward checkpoints and results from the previous stage, if one is attached.
for prev in sorted(p for p in os.listdir("/kaggle/input") if p.startswith("halo-stage")):
    for sub in ("checkpoints", "results", "figures"):
        s = f"/kaggle/input/{prev}/{sub}"
        if os.path.isdir(s):
            os.makedirs(f"/kaggle/working/{sub}", exist_ok=True)
            for f in os.listdir(s):
                shutil.copy2(os.path.join(s, f), f"/kaggle/working/{sub}/{f}")

from halo.io import environment_manifest
env = environment_manifest()
print("ENVIRONMENT (observed, not assumed):")
for k, v in env.items():
    print(f"  {k:24s} {v}")


In [ ]:
from halo.cli import prepare, main
import argparse, numpy as np
from halo.regimes import RegimeMiner
from halo.risk import AssociationRisk
from halo.protocol import rolling_origin_folds
from halo.config import CFG

args = argparse.Namespace(synthetic=False, entities=6000, rows=None,
                          seeds=list(CFG.seeds), delta=None, uid=CFG.uid_variant,
                          model="lightgbm", tuning_budget=0, no_cache=False)
df, entity, is_index, _ = prepare(args)
folds = rolling_origin_folds(df, entity, latency_delta=CFG.headline_delta,
                             entity_disjoint=True)
print(f"CEP folds: {len(folds)}")
for f in folds:
    print("  ", f.meta())

miner = RegimeMiner().fit(df.iloc[folds[0].train_idx])
print("\nBlock C:", miner.summary())
print("Largest column blocks:")
for name, members in sorted(miner.column_blocks_.items(),
                            key=lambda kv: -len(kv[1]))[:8]:
    print(f"   {len(members):4d} cols  e.g. {members[:6]}")
